In [2]:
import json
import glob
import pandas as pd
from pathlib import Path


In [3]:

sweep_dir = "/home/ucloud/BioASQ13B/phaseB/dev/sweep_outputs/submission1-0.64_hydrated"

rows = []
for path in sorted(glob.glob(f"{sweep_dir}/*.json")):
    name = Path(path).stem
    data = json.load(open(path))
    total = len(data)
    failures = sum(1 for v in data.values() if not v["valid"])
    rows.append({"run": name, "total": total, "failures": failures, "failure_rate": failures / total if total else 0})

df = pd.DataFrame(rows).sort_values("failures", ascending=False)
df["failure_rate"] = df["failure_rate"].map("{:.1%}".format)
df

,run,total,failures,failure_rate
4,claude-haiku-4-5-20251001_abstracts_10_5,80,80,100.0%
14,gemini-2.0-flash-001_abstracts_10_7,80,80,100.0%
7,claude-haiku-4-5-20251001_abstracts_5_6,80,80,100.0%
6,claude-haiku-4-5-20251001_abstracts_5_5,80,80,100.0%
5,claude-haiku-4-5-20251001_abstracts_10_6,80,80,100.0%
28,gemini-2.0-flash-001_abstracts_5_7,80,80,100.0%
41,medgemma-27b-text-it_abstracts_10_5,80,80,100.0%
21,gemini-2.0-flash-001_abstracts_3_7,80,80,100.0%
43,medgemma-27b-text-it_abstracts_5_5,80,74,92.5%
0,NVIDIA-Nemotron-3-Nano-30B-A3B-BF16_abstracts_...,80,58,72.5%


In [5]:
import json
import glob
import pandas as pd
from pathlib import Path

sweep_dir = "/home/ucloud/BioASQ13B/phaseB/dev/sweep_outputs/submission1-0.64_hydrated"

# Load all runs
rows = []
for path in sorted(glob.glob(f"{sweep_dir}/*.json")):
    name = Path(path).stem
    # Extract model name (everything before _abstracts_)
    model = name.split("_abstracts_")[0]
    data = json.load(open(path))
    total = len(data)
    failures = sum(1 for v in data.values() if not v["valid"])
    rows.append({"run": name, "model": model, "total": total, "failures": failures, "data": data})

df = pd.DataFrame(rows)

# Pick best run per model (fewest failures)
best_per_model = df.loc[df.groupby("model")["failures"].idxmin()].reset_index(drop=True)
print("Best run per model:")
print(best_per_model[["model", "run", "failures", "total"]].to_string())

# Get common question IDs across all best runs
all_ids = [set(row["data"].keys()) for _, row in best_per_model.iterrows()]
common_ids = sorted(set.intersection(*all_ids))
print(f"\n{len(common_ids)} questions in common across all best runs")

# Show numbered sample
SAMPLE_N = 10
sample_ids = common_ids[:SAMPLE_N]

for i, qid in enumerate(sample_ids, 1):
    print(f"\n{'='*70}")
    print(f"Q{i}. ID: {qid}")
    for _, row in best_per_model.iterrows():
        entry = row["data"].get(qid, {})
        text = entry.get("text", "").strip() or "[EMPTY/FAILED]"
        valid = entry.get("valid", False)
        print(f"  [{row['model']}] (valid={valid})")
        print(f"    {text}")

Best run per model:
                                 model                                                run  failures  total
0  NVIDIA-Nemotron-3-Nano-30B-A3B-BF16  NVIDIA-Nemotron-3-Nano-30B-A3B-BF16_abstracts_5_6         2     80
1            claude-haiku-4-5-20251001           claude-haiku-4-5-20251001_abstracts_10_5        80     80
2                 gemini-2.0-flash-001                gemini-2.0-flash-001_abstracts_10_1         0     80
3                     gemini-2.5-flash                     gemini-2.5-flash_abstracts_5_6         0     80
4                       gemma-3-27b-it                      gemma-3-27b-it_abstracts_10_6         0     80
5                        grok-4.1-fast                       grok-4.1-fast_abstracts_10_5         0     80
6                 medgemma-27b-text-it                 medgemma-27b-text-it_abstracts_5_6         0     80
7                            qwen3-32b                           qwen3-32b_abstracts_10_6        42     80

80 questions in 